# set up

In [1]:
#set up
import spacy
import pandas as pd


In [2]:
from google.colab import drive
drive.mount('/content/drive')
data = ('/content/drive/My Drive/Colab Notebooks/SocialMediaPosts/MatchTextLength1204/02DataForAnalysis/')
result = ('/content/drive/My Drive/Colab Notebooks/SocialMediaPosts/MatchTextLength1204/03Results/')
brief = ('/content/drive/My Drive/Colab Notebooks/SocialMediaPosts/MatchTextLength1204/04DataForR/')

Mounted at /content/drive


# define the funtion: spacy_TTR

 Analyzes a single text string to compute sentence count, word count, unique word count, and Type-Token Ratio (TTR).


In [3]:
# Load the spaCy model
nlp = spacy.load("en_core_web_sm")


In [7]:
def spacy_TTR(text):

    if not isinstance(text, str):
        return pd.Series([0, 0, 0, 0.0])  # Handle non-string inputs gracefully

    # Process the text using spaCy
    doc = nlp(text)

    # Calculate sentence count
    sent_count = len(list(doc.sents)) #doc.sents:SpaCy uses its sentence boundary detection (SBD) algorithm to split the text into sentences.

    # Extract words (excluding punctuation and spaces)
    words = [token.text.lower() for token in doc if token.is_alpha]
    word_count = len(words)

    # Calculate unique word count
    unique_word_count = len(set(words))

    # Calculate Type-Token Ratio (TTR)
    TTR = unique_word_count / word_count if word_count > 0 else 0

    # Return the results as a pandas Series
    return pd.Series([sent_count, word_count, unique_word_count, TTR])

**Let us see an example**

In [8]:
# Sample DataFrame with a 'clean_text' column
data = {'clean_text': [
        "I love using Python! It's so versatile.",
        "Data science and machine learning are fascinating.",
        "SpaCy is great for NLP tasks."]}

df = pd.DataFrame(data)

In [9]:
# Apply the compute_TTR function to the 'clean_text' column and add new columns to the DataFrame
df[['sent_count', 'word_count', 'unique_word_count', 'TTR']] = df['clean_text'].apply(spacy_TTR)
df

,clean_text,sent_count,word_count,unique_word_count,TTR
0,I love using Python! It's so versatile.,2.0,7.0,7.0,1.0
1,Data science and machine learning are fascinat...,1.0,7.0,7.0,1.0
2,SpaCy is great for NLP tasks.,1.0,6.0,6.0,1.0


# define the function: syntactic_complexity_metric

    Analyze a single text entry to compute syntactic complexity metrics:
    - Total dependents
    - Dependents per clause
    - Coordinate phrases per clause
    - Complex nominals per clause

**Degree of sophistication (spacy)**

- the number of dependents per clause (amount of subordination),
- the number of coordinate phrases per clause (amount of coordination),
- the number of complex nominals per clause (degree of phrasal sophistication).

All the syntactic indices will be computed using spaCy.

In [10]:
nlp = spacy.load('en_core_web_sm')

In [11]:
def syntactic_complexity_metrics(text):

    if not isinstance(text, str):
        return pd.Series([0, 0, 0, 0, 0, 0])  # Handle non-string inputs gracefully

    # Process the text using spaCy
    doc = nlp(text)

    sen_count = 0     # Total number of sentences (treated as clauses)
    total_dependents = 0
    total_coordinate_phrases = 0
    total_complex_nominals = 0

    # Iterate over sentences in the text
    for sent in doc.sents:
        sen_count += 1

        # Initialize counters for the current sentence
        sent_dependents = 0
        sent_coord_phrases = 0
        sent_complex_nominals = 0

        for token in sent:
            # 1. Count dependents per clause
            if token.dep_ in {'advcl', 'csubj', 'ccomp', 'acl', 'xcomp', 'relcl'}:
                sent_dependents += 1

            # 2. Count coordinate phrases per clause
            if token.dep_ in {'cc', 'conj'}:
                sent_coord_phrases += 1

            # 3. Count complex nominals per clause
            if token.dep_ in {"nsubj", "dobj", "pobj", "iobj", "nmod"}:
                if any(child.dep_ in {"amod", "poss", "compound"} for child in token.children):
                    sent_complex_nominals += 1

        total_dependents += sent_dependents
        total_coordinate_phrases += sent_coord_phrases
        total_complex_nominals += sent_complex_nominals

    # Avoid division by zero (if no sentences are found)
    if sen_count == 0:
        return pd.Series([0, 0, 0, 0, 0, 0])

    # Calculate per-clause averages
    dependents_per_clause = total_dependents / sen_count
    coord_phrases_per_clause = total_coordinate_phrases / sen_count
    complex_nominals_per_clause = total_complex_nominals / sen_count

    # Return results as a pandas Series
    return pd.Series([
        total_dependents, dependents_per_clause,
        total_coordinate_phrases, coord_phrases_per_clause,
        total_complex_nominals, complex_nominals_per_clause
    ])

In [12]:
df[['total_dependents', 'dependents_per_clause',
    'total_coord_phrases', 'coord_phrases_per_clause',
    'total_complex_nominals', 'complex_nominals_per_clause']] = df['clean_text'].apply(syntactic_complexity_metrics)
df

,clean_text,sent_count,word_count,unique_word_count,TTR,total_dependents,dependents_per_clause,total_coord_phrases,coord_phrases_per_clause,total_complex_nominals,complex_nominals_per_clause
0,I love using Python! It's so versatile.,2.0,7.0,7.0,1.0,1.0,0.5,0.0,0.0,0.0,0.0
1,Data science and machine learning are fascinat...,1.0,7.0,7.0,1.0,0.0,0.0,2.0,2.0,1.0,1.0
2,SpaCy is great for NLP tasks.,1.0,6.0,6.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0


# AD (social media group)

In [13]:
#read the data
ad_social = pd.read_csv(result + 'AD_social_result.csv', engine='python')
ad_social.head(2)

,idx,creation_time,id,is_branded_content,lang,link_attachment.caption,link_attachment.description,link_attachment.link,link_attachment.name,media_type,...,surface.name,surface.username,text,clean_text,cluster,VADER_neg,VADER_neu,VADER_pos,VADER_compound,VADER_sentiment
0,0,2024-09-16T08:00:05-07:00,1.194189e+15,False,en,NaN,NaN,NaN,NaN,albums,...,Alzheimer's Society,alzheimerssocietyuk,Shocking new data has shown that people living...,shocking new data has shown that people living...,3.0,0.089,0.847,0.064,-0.9070,Negative
1,1,2024-09-14T13:00:08-07:00,8.765100e+14,False,en,NaN,NaN,NaN,NaN,albums,...,Alzheimer's Society,alzheimerssocietyuk,‘I never imagined dementia impacting our lives...,‘i never imagined dementia impacting our lives...,3.0,0.020,0.815,0.165,0.9947,Positive


In [14]:
# apply function to calculate TTR
ad_social[['sent_count', 'word_count', 'unique_word_count', 'TTR']] = ad_social['clean_text'].apply(spacy_TTR)
ad_social.head(2)


,idx,creation_time,id,is_branded_content,lang,link_attachment.caption,link_attachment.description,link_attachment.link,link_attachment.name,media_type,...,cluster,VADER_neg,VADER_neu,VADER_pos,VADER_compound,VADER_sentiment,sent_count,word_count,unique_word_count,TTR
0,0,2024-09-16T08:00:05-07:00,1.194189e+15,False,en,NaN,NaN,NaN,NaN,albums,...,3.0,0.089,0.847,0.064,-0.9070,Negative,25.0,363.0,197.0,0.542700
1,1,2024-09-14T13:00:08-07:00,8.765100e+14,False,en,NaN,NaN,NaN,NaN,albums,...,3.0,0.020,0.815,0.165,0.9947,Positive,14.0,294.0,177.0,0.602041


In [15]:
# apply function to calculate Degree of sophistication
ad_social[['total_dependents', 'dependents_per_clause',
    'total_coord_phrases', 'coord_phrases_per_clause',
    'total_complex_nominals', 'complex_nominals_per_clause']] = ad_social['clean_text'].apply(syntactic_complexity_metrics)
ad_social.head(2)

,idx,creation_time,id,is_branded_content,lang,link_attachment.caption,link_attachment.description,link_attachment.link,link_attachment.name,media_type,...,sent_count,word_count,unique_word_count,TTR,total_dependents,dependents_per_clause,total_coord_phrases,coord_phrases_per_clause,total_complex_nominals,complex_nominals_per_clause
0,0,2024-09-16T08:00:05-07:00,1.194189e+15,False,en,NaN,NaN,NaN,NaN,albums,...,25.0,363.0,197.0,0.542700,28.0,1.120000,26.0,1.04,28.0,1.120000
1,1,2024-09-14T13:00:08-07:00,8.765100e+14,False,en,NaN,NaN,NaN,NaN,albums,...,14.0,294.0,177.0,0.602041,25.0,1.785714,28.0,2.00,24.0,1.714286


In [16]:
# save the results to a new cvs. file
ad_social.to_csv(result + 'AD_social_result.csv', index=False)

let us also save the result to the brief result column

In [17]:
df1 = pd.read_csv(brief + 'AD_social_brief_result.csv')
df1.head(2)

,idx,id,clean_text,VADER_neg,VADER_neu,VADER_pos,VADER_compound,VADER_sentiment
0,0,1.194189e+15,shocking new data has shown that people living...,0.089,0.847,0.064,-0.9070,Negative
1,1,8.765100e+14,‘i never imagined dementia impacting our lives...,0.020,0.815,0.165,0.9947,Positive


In [18]:
df1[['sent_count', 'word_count', 'unique_word_count', 'TTR']] = df1['clean_text'].apply(spacy_TTR)
df1.head(2)

,idx,id,clean_text,VADER_neg,VADER_neu,VADER_pos,VADER_compound,VADER_sentiment,sent_count,word_count,unique_word_count,TTR
0,0,1.194189e+15,shocking new data has shown that people living...,0.089,0.847,0.064,-0.9070,Negative,25.0,363.0,197.0,0.542700
1,1,8.765100e+14,‘i never imagined dementia impacting our lives...,0.020,0.815,0.165,0.9947,Positive,14.0,294.0,177.0,0.602041


In [19]:
df1[['total_dependents', 'dependents_per_clause',
    'total_coord_phrases', 'coord_phrases_per_clause',
    'total_complex_nominals', 'complex_nominals_per_clause']] = df1['clean_text'].apply(syntactic_complexity_metrics)
df1.head(2)

,idx,id,clean_text,VADER_neg,VADER_neu,VADER_pos,VADER_compound,VADER_sentiment,sent_count,word_count,unique_word_count,TTR,total_dependents,dependents_per_clause,total_coord_phrases,coord_phrases_per_clause,total_complex_nominals,complex_nominals_per_clause
0,0,1.194189e+15,shocking new data has shown that people living...,0.089,0.847,0.064,-0.9070,Negative,25.0,363.0,197.0,0.542700,28.0,1.120000,26.0,1.04,28.0,1.120000
1,1,8.765100e+14,‘i never imagined dementia impacting our lives...,0.020,0.815,0.165,0.9947,Positive,14.0,294.0,177.0,0.602041,25.0,1.785714,28.0,2.00,24.0,1.714286


In [20]:
# save the results to a new cvs. file
df1.to_csv(brief + 'AD_social_brief_result.csv', index=False)

# Health (social media group)

In [21]:
#read the data
health_social = pd.read_csv(result + 'Health_social_result.csv',engine='python')
health_social.head(2)

,idx,statistics.like_count,link_attachment.caption,statistics.love_count,post_owner.username,post_owner.name,modified_time,statistics.haha_count,surface.id,lang,...,statistics.reaction_count,statistics.wow_count,surface.name,statistics.sad_count,clean_text,VADER_neg,VADER_neu,VADER_pos,VADER_compound,VADER_sentiment
0,0,67.0,aarp.org,0.0,AARP,AARP,2024-10-02T10:48:06-07:00,0.0,6.971043e+14,en,...,67.0,0.0,AARP,0.0,aarp volunteers gathered with user experience ...,0.00,0.793,0.207,0.8807,Positive
1,1,96.0,aarp.org,3.0,AARP,AARP,2024-10-02T10:53:20-07:00,2.0,6.971043e+14,en,...,102.0,0.0,AARP,0.0,taraji p. henson stars in the newly launched p...,0.06,0.750,0.190,0.9393,Positive


In [22]:
health_social[['sent_count', 'word_count', 'unique_word_count', 'TTR']] = health_social['clean_text'].apply(spacy_TTR)
health_social.head(2)

,idx,statistics.like_count,link_attachment.caption,statistics.love_count,post_owner.username,post_owner.name,modified_time,statistics.haha_count,surface.id,lang,...,clean_text,VADER_neg,VADER_neu,VADER_pos,VADER_compound,VADER_sentiment,sent_count,word_count,unique_word_count,TTR
0,0,67.0,aarp.org,0.0,AARP,AARP,2024-10-02T10:48:06-07:00,0.0,6.971043e+14,en,...,aarp volunteers gathered with user experience ...,0.00,0.793,0.207,0.8807,Positive,2.0,49.0,41.0,0.836735
1,1,96.0,aarp.org,3.0,AARP,AARP,2024-10-02T10:53:20-07:00,2.0,6.971043e+14,en,...,taraji p. henson stars in the newly launched p...,0.06,0.750,0.190,0.9393,Positive,4.0,98.0,72.0,0.734694


In [23]:
health_social[['total_dependents', 'dependents_per_clause',
    'total_coord_phrases', 'coord_phrases_per_clause',
    'total_complex_nominals', 'complex_nominals_per_clause']] = health_social['clean_text'].apply(syntactic_complexity_metrics)
health_social.head(2)

,idx,statistics.like_count,link_attachment.caption,statistics.love_count,post_owner.username,post_owner.name,modified_time,statistics.haha_count,surface.id,lang,...,sent_count,word_count,unique_word_count,TTR,total_dependents,dependents_per_clause,total_coord_phrases,coord_phrases_per_clause,total_complex_nominals,complex_nominals_per_clause
0,0,67.0,aarp.org,0.0,AARP,AARP,2024-10-02T10:48:06-07:00,0.0,6.971043e+14,en,...,2.0,49.0,41.0,0.836735,5.0,2.50,6.0,3.0,6.0,3.0
1,1,96.0,aarp.org,3.0,AARP,AARP,2024-10-02T10:53:20-07:00,2.0,6.971043e+14,en,...,4.0,98.0,72.0,0.734694,3.0,0.75,10.0,2.5,12.0,3.0


In [24]:
# save the results to the cvs. file
health_social.to_csv(result + 'Health_social_result.csv', index=False)

let us also save the results to another cvs fule

In [25]:
df2 = pd.read_csv(brief + 'Health_social_brief_result.csv')
df2.head(2)

,idx,id,clean_text,VADER_neg,VADER_neu,VADER_pos,VADER_compound,VADER_sentiment
0,0,1.087023e+15,aarp volunteers gathered with user experience ...,0.00,0.793,0.207,0.8807,Positive
1,1,1.527138e+15,taraji p. henson stars in the newly launched p...,0.06,0.750,0.190,0.9393,Positive


In [26]:
df2[['sent_count', 'word_count', 'unique_word_count', 'TTR']] = df2['clean_text'].apply(spacy_TTR)
df2.head(2)

,idx,id,clean_text,VADER_neg,VADER_neu,VADER_pos,VADER_compound,VADER_sentiment,sent_count,word_count,unique_word_count,TTR
0,0,1.087023e+15,aarp volunteers gathered with user experience ...,0.00,0.793,0.207,0.8807,Positive,2.0,49.0,41.0,0.836735
1,1,1.527138e+15,taraji p. henson stars in the newly launched p...,0.06,0.750,0.190,0.9393,Positive,4.0,98.0,72.0,0.734694


In [27]:
df2[['total_dependents', 'dependents_per_clause',
    'total_coord_phrases', 'coord_phrases_per_clause',
    'total_complex_nominals', 'complex_nominals_per_clause']] = df2['clean_text'].apply(syntactic_complexity_metrics)
df2.head(2)

,idx,id,clean_text,VADER_neg,VADER_neu,VADER_pos,VADER_compound,VADER_sentiment,sent_count,word_count,unique_word_count,TTR,total_dependents,dependents_per_clause,total_coord_phrases,coord_phrases_per_clause,total_complex_nominals,complex_nominals_per_clause
0,0,1.087023e+15,aarp volunteers gathered with user experience ...,0.00,0.793,0.207,0.8807,Positive,2.0,49.0,41.0,0.836735,5.0,2.50,6.0,3.0,6.0,3.0
1,1,1.527138e+15,taraji p. henson stars in the newly launched p...,0.06,0.750,0.190,0.9393,Positive,4.0,98.0,72.0,0.734694,3.0,0.75,10.0,2.5,12.0,3.0


In [28]:
# save the results to a new cvs. file
df2.to_csv(brief + 'Health_social_brief_result.csv', index=False)

# AD (clinical group)

In [29]:
#read the data
ad_clinical = pd.read_csv(result + 'AD_clinical_result.csv',engine='python')
ad_clinical.head(2)

,idx,PAR,corpus,diag,age,gender,clean_text,VADER_neg,VADER_neu,VADER_pos,VADER_compound,VADER_sentiment
0,0,dementia-001-2,He_Hinzen,ModerateAD,59,male,mhm . there's a young boy going in a cookie ja...,0.029,0.951,0.020,-0.0258,Neutral
1,1,dementia-003-0,He_Hinzen,MildAD,56,male,here's a cookie jar . and the lid is off the c...,0.013,0.916,0.071,0.9161,Positive


In [30]:
ad_clinical[['sent_count', 'word_count', 'unique_word_count', 'TTR']] = ad_clinical['clean_text'].apply(spacy_TTR)
ad_clinical.head(2)

,idx,PAR,corpus,diag,age,gender,clean_text,VADER_neg,VADER_neu,VADER_pos,VADER_compound,VADER_sentiment,sent_count,word_count,unique_word_count,TTR
0,0,dementia-001-2,He_Hinzen,ModerateAD,59,male,mhm . there's a young boy going in a cookie ja...,0.029,0.951,0.020,-0.0258,Neutral,12.0,110.0,59.0,0.536364
1,1,dementia-003-0,He_Hinzen,MildAD,56,male,here's a cookie jar . and the lid is off the c...,0.013,0.916,0.071,0.9161,Positive,20.0,189.0,87.0,0.460317


In [31]:
ad_clinical[['total_dependents', 'dependents_per_clause',
    'total_coord_phrases', 'coord_phrases_per_clause',
    'total_complex_nominals', 'complex_nominals_per_clause']] = ad_clinical['clean_text'].apply(syntactic_complexity_metrics)
ad_clinical.head(2)

,idx,PAR,corpus,diag,age,gender,clean_text,VADER_neg,VADER_neu,VADER_pos,...,sent_count,word_count,unique_word_count,TTR,total_dependents,dependents_per_clause,total_coord_phrases,coord_phrases_per_clause,total_complex_nominals,complex_nominals_per_clause
0,0,dementia-001-2,He_Hinzen,ModerateAD,59,male,mhm . there's a young boy going in a cookie ja...,0.029,0.951,0.020,...,12.0,110.0,59.0,0.536364,11.0,0.916667,9.0,0.75,3.0,0.25
1,1,dementia-003-0,He_Hinzen,MildAD,56,male,here's a cookie jar . and the lid is off the c...,0.013,0.916,0.071,...,20.0,189.0,87.0,0.460317,16.0,0.800000,20.0,1.00,3.0,0.15


In [32]:
# save the results to a new cvs. file
ad_clinical.to_csv(result + 'AD_clinical_result.csv', index=False)

# Health (clinical group)

In [33]:
#read the data
health_clinical = pd.read_csv(result + 'Health_clinical_result.csv',engine='python')
health_clinical.head(2)

,idx,PAR,corpus,diag,age,gender,clean_text,VADER_neg,VADER_neu,VADER_pos,VADER_compound,VADER_sentiment
0,0,control-002-0,He_Hinzen,Control,58,female,the scene is in the kitchen . the mother is wi...,0.01,0.925,0.064,0.7579,Positive
1,1,control-013-0,He_Hinzen,Control,62,female,somebody's getting cookies out of the cookie j...,0.02,0.980,0.000,-0.2547,Negative


In [34]:
health_clinical[['sent_count', 'word_count', 'unique_word_count', 'TTR']] = health_clinical['clean_text'].apply(spacy_TTR)
health_clinical.head(2)

,idx,PAR,corpus,diag,age,gender,clean_text,VADER_neg,VADER_neu,VADER_pos,VADER_compound,VADER_sentiment,sent_count,word_count,unique_word_count,TTR
0,0,control-002-0,He_Hinzen,Control,58,female,the scene is in the kitchen . the mother is wi...,0.01,0.925,0.064,0.7579,Positive,18.0,133.0,80.0,0.601504
1,1,control-013-0,He_Hinzen,Control,62,female,somebody's getting cookies out of the cookie j...,0.02,0.980,0.000,-0.2547,Negative,12.0,88.0,56.0,0.636364


In [35]:
health_clinical[['total_dependents', 'dependents_per_clause',
    'total_coord_phrases', 'coord_phrases_per_clause',
    'total_complex_nominals', 'complex_nominals_per_clause']] = health_clinical['clean_text'].apply(syntactic_complexity_metrics)
health_clinical.head(2)

,idx,PAR,corpus,diag,age,gender,clean_text,VADER_neg,VADER_neu,VADER_pos,...,sent_count,word_count,unique_word_count,TTR,total_dependents,dependents_per_clause,total_coord_phrases,coord_phrases_per_clause,total_complex_nominals,complex_nominals_per_clause
0,0,control-002-0,He_Hinzen,Control,58,female,the scene is in the kitchen . the mother is wi...,0.01,0.925,0.064,...,18.0,133.0,80.0,0.601504,8.0,0.444444,3.0,0.166667,5.0,0.277778
1,1,control-013-0,He_Hinzen,Control,62,female,somebody's getting cookies out of the cookie j...,0.02,0.980,0.000,...,12.0,88.0,56.0,0.636364,4.0,0.333333,6.0,0.500000,2.0,0.166667


In [36]:
# save the results to the cvs. file
health_clinical.to_csv(result + 'Health_clinical_result.csv', index=False)